In [ ]:
# Main import block
import warnings
from contextlib import contextmanager
from datetime import date

import pandas as pd
import requests

warnings.filterwarnings("ignore")

In [ ]:
################################################################
##########################  SHMET ##############################
################################################################

async def shmet_optimized_async():
    url = "https://en.shmet.com/api/rest/enweb/spot/getSpotPrice?code=baseMetal&size=10&currentLength=0"
    
    try:
        responce = requests.get(url)

        day_df = pd.DataFrame(responce.json()["data"])
        day_df["date"] = date.today()
        cooper_row = day_df[day_df["name"].str.contains("cu", case=False)]

        result = cooper_row[["date", "middle", "unit"]]
        result["date"] = pd.to_datetime(result["date"])
        result = result.rename(columns={"middle": "price"})

        hist_data = pd.read_excel("./data/shmet_historical.xlsx", index_col=0)

        new_df = pd.concat([result, hist_data], axis=0).reset_index(drop=True)
        new_df.drop_duplicates(inplace=True)

        with pd.ExcelWriter(
            "./data/shmet_historical.xlsx",
            date_format="YYYY-MM-DD",
            datetime_format="YYYY-MM-DD",
        ) as writer:
            new_df.to_excel(writer, sheet_name="SHMET")

        final = pd.read_excel("./data/shmet_historical.xlsx", index_col=0)
        final.drop_duplicates(inplace=True)

        with pd.ExcelWriter(
            "./data/shmet_historical.xlsx",
            date_format="YYYY-MM-DD",
            datetime_format="YYYY-MM-DD",
        ) as writer:
            final.to_excel(writer, sheet_name="SHMET")

        print("SHMET is done!!!")
        # display(final.head(10))
    
    except Exception as error:
        print(error)
